# MNLP Homework 1 - Semantic Search  
Semantic search is an advanced search technique that focuses on understanding the user's intent and the contextual meaning of a query, rather than just matching exact keywords. It uses AI to find results that are conceptually related to what you asked, even if you use different words or synonyms.

In [ ]:
# Import of dataset from huggingface
from datasets import load_dataset
import datasets

# Suppress info/debug logging, unnecessary output
datasets.logging.set_verbosity_error()

mnlp_dataset = load_dataset("sapienzanlp-course-materials/hw-mnlp-2026")

Dataset is composed of 3 sets:  
- Train
- Test
- Blind
  
Only Train can be used to Train the model and Test to evaluate its performances.  
  
Available columns that can be used:  
- 'query',
- 'query_id',
- 'candidate_chunks',
- 'n_candidates',
- 'answer',
- 'answer_pos'

In [ ]:
# Load train, test and blind datasets
train_data = mnlp_dataset['train']
test_data = mnlp_dataset['test']
blind_data = mnlp_dataset['blind']

print(f"Train set size: {len(train_data)}")
print(f"Test set size: {len(test_data)}")
print(f"Blind set size: {len(blind_data)}")

print(f"\nFirst training example:")
print(f"Query: {train_data[0]['query']}")
print(f"Number of candidates: {train_data[0]['n_candidates']}")
print(f"Answer: {train_data[0]['answer']}")
print(f"Answer position: {train_data[0]['answer_pos']}")

## Task [B1]  
Implement the two baseline semantic search systems:  
- distilbert-base-uncased
- all-MiniLM-L6-v2

In [ ]:
# Import required libraries
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from tqdm import tqdm

# Install required packages if needed
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers", "torch", "-q"])

# GPU usage check (doublecheck if we started with right gpu)
print("Colab actual device: " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Evaluate semantic search system using a given HuggingFace transformer model.
Uses modular steps for clarity:
1) Transform candidate_chunks into fixed-size dense vector representation (sentence embeddings)
2) Encode query into fixed-size dense vector representation
3) Use cosine similarity metrics to find chunk closest to question
4) Use Hit@k metric to evaluate results

In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


def evaluate_semantic_search(model_name, dataset, split_name="test"):
    print(f"\n{'='*60}")
    print(f"Evaluating model: {model_name}")
    print(f"Dataset: {split_name} ({len(dataset)} samples)")
    print(f"{'='*60}\n")
    
    # Load model and tokenizer
    print(f"[1/4] Loading model {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()
    print(f"✓ Model ready on {device}\n")
    
    # Metrics to track
    hits_at_1 = 0
    hits_at_3 = 0
    hits_at_5 = 0
    total_samples = len(dataset)
    
    print(f"[2/4] Processing {total_samples} queries...")
    print(f"Progress | Hit@1 | Hit@3 | Hit@5")
    print("-" * 35)
    
    # Process each query
    for idx, example in enumerate(tqdm(dataset, desc="Processing queries")):
        query = example['query']
        candidate_chunks = example['candidate_chunks']
        answer_pos = example['answer_pos']
        
        # Step 1 & 2: Encode texts into embeddings
        query_embedding = encode_texts_to_embeddings([query], tokenizer, model, device, mean_pooling)[0]
        chunk_embeddings = encode_texts_to_embeddings(candidate_chunks, tokenizer, model, device, mean_pooling)
        
        # Step 3: Compute cosine similarity and get ranking
        ranked_indices, similarities = compute_cosine_similarity_ranking(query_embedding, chunk_embeddings)
        
        # Step 4: Compute Hit@k metrics
        hit_metrics = compute_hitk_metrics(ranked_indices, answer_pos, k_values=[1, 3, 5])
        hits_at_1 += hit_metrics[1]
        hits_at_3 += hit_metrics[3]
        hits_at_5 += hit_metrics[5]
        
        # Print progress every 10% of dataset
        if (idx + 1) % max(1, total_samples // 10) == 0:
            progress_pct = (idx + 1) / total_samples * 100
            h1 = (hits_at_1 / (idx + 1))
            h3 = (hits_at_3 / (idx + 1))
            h5 = (hits_at_5 / (idx + 1))
            print(f"{progress_pct:5.1f}%  | {h1:.3f} | {h3:.3f} | {h5:.3f}")
    
    # Calculate final metrics
    hit_at_1 = hits_at_1 / total_samples
    hit_at_3 = hits_at_3 / total_samples
    hit_at_5 = hits_at_5 / total_samples
    
    print(f"100.0%  | {hit_at_1:.3f} | {hit_at_3:.3f} | {hit_at_5:.3f}")
    print(f"\n[3/4] Evaluation complete!")
    print(f"[4/4] Results ready\n")
    
    return {
        'model': model_name,
        'hit@1': hit_at_1,
        'hit@3': hit_at_3,
        'hit@5': hit_at_5,
    }

Transform texts into fixed-size dense vector representation (sentence embeddings)
and memorize them.

Args:
    texts: List of text strings
    tokenizer: Tokenizer from transformers
    model: Model from transformers
    device: torch device (cuda or cpu)
    mean_pooling_fn: Function to perform mean pooling

Returns:
    Normalized embeddings as numpy array

In [ ]:
# Step 1 & 2: Encode texts into fixed-size dense vector representations
def encode_texts_to_embeddings(texts, tokenizer, model, device, mean_pooling_fn):
    encoded = tokenizer(texts, padding=True, truncation=True, return_tensors='pt', max_length=256)
    encoded = {k: v.to(device) for k, v in encoded.items()}
    
    with torch.no_grad():
        model_output = model(**encoded)
    
    sentence_embeddings = mean_pooling_fn(model_output, encoded['attention_mask'])
    return torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1).cpu().numpy()

Compute cosine similarity between query and candidate chunks,
and return ranked indices sorted by similarity (highest first).

Args:
    query_embedding: Single query embedding (1D numpy array)
    chunk_embeddings: Chunk embeddings (2D numpy array)

Returns:
    ranked_indices: Array of indices sorted by similarity (highest first)
    similarities: Array of similarity scores

In [ ]:
# Step 3: Use cosine similarity metrics to find chunk closest to question
def compute_cosine_similarity_ranking(query_embedding, chunk_embeddings):
    similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]
    ranked_indices = np.argsort(similarities)[::-1]
    return ranked_indices, similarities

Compute Hit@k metrics.
Hit@k = 1 if the correct answer appears in top-k results, 0 otherwise.

Args:
    ranked_indices: Array of indices ranked by similarity (highest first)
    answer_pos: Index position of the correct answer
    k_values: List of k values to compute (default: [1, 3, 5])

Returns:
    Dictionary with Hit@k results for each k value

In [ ]:
# Step 4: Use Hit@k metric to evaluate results
def compute_hitk_metrics(ranked_indices, answer_pos, k_values=[1, 3, 5]):
    hits = {k: 0 for k in k_values}
    
    # Find position of correct answer in the ranking
    for rank, idx_pos in enumerate(ranked_indices):
        if idx_pos == answer_pos:
            for k in k_values:
                if rank < k:
                    hits[k] = 1
            break
    
    return hits

In [ ]:
# Baseline 1: distilbert-base-uncased
model_1_results = evaluate_semantic_search('distilbert-base-uncased', test_data, 'test')

print(f"\n{'='*60}")
print(f"Results for {model_1_results['model']}")
print(f"{'='*60}")
print(f"Hit@1:  {model_1_results['hit@1']:.4f}")
print(f"Hit@3:  {model_1_results['hit@3']:.4f}")
print(f"Hit@5:  {model_1_results['hit@5']:.4f}")

In [ ]:
# Baseline 2: all-MiniLM-L6-v2
model_2_results = evaluate_semantic_search('sentence-transformers/all-MiniLM-L6-v2', test_data, 'test')

print(f"\n{'='*60}")
print(f"Results for {model_2_results['model']}")
print(f"{'='*60}")
print(f"Hit@1:  {model_2_results['hit@1']:.4f}")
print(f"Hit@3:  {model_2_results['hit@3']:.4f}")
print(f"Hit@5:  {model_2_results['hit@5']:.4f}")

## Task [B2]
Fine-tune DistilBERT using supervised triplet loss to improve semantic search performance. 

Fine-tuning strategy:
1) Create triplet dataset: (query, positive_chunk, negative_chunk)
2) Train using triplet loss to minimize distance between query and correct answer
3) Maximize distance between query and random other chunks
4) Evaluate on test set and compare against baseline

In [ ]:
# Install sentence-transformers for triplet loss support
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "sentence-transformers", "-q"])

from sentence_transformers import SentenceTransformer, losses, models, InputExample
from sentence_transformers.losses import TripletLoss
from torch.utils.data import DataLoader
import torch

print("B2 needed imports done.")

In [ ]:
# Create triplet dataset from training data
import random

class TripletDataset:
    """
    Dataset that generates triplets: (query, positive_chunk, negative_chunk)
    Returns simple tuples instead of InputExample objects for PyTorch DataLoader
    """
    def __init__(self, dataset):
        self.triplets = []
        
        print("[1/2] Creating triplet dataset...")
        for example in dataset:
            query = example['query']
            candidates = example['candidate_chunks']
            answer_pos = example['answer_pos']
            positive_chunk = candidates[answer_pos]
            
            # Select a random negative (any chunk that is NOT the answer)
            negative_positions = [i for i in range(len(candidates)) if i != answer_pos]
            negative_pos = random.choice(negative_positions)
            negative_chunk = candidates[negative_pos]
            
            # Store as simple tuple (not InputExample)
            self.triplets.append((query, positive_chunk, negative_chunk))
        
        print(f"✓ Created {len(self.triplets)} triplets from {len(dataset)} samples")
        print(f"  Example triplet structure:")
        print(f"  - Anchor (Query): {self.triplets[0][0][:60]}...")
        print(f"  - Positive: {self.triplets[0][1][:60]}...")
        print(f"  - Negative: {self.triplets[0][2][:60]}...")
    
    def __len__(self):
        return len(self.triplets)
    
    def __getitem__(self, idx):
        return self.triplets[idx]

# Create triplet dataset
triplet_dataset = TripletDataset(train_data)

# Create DataLoader with custom collate function
def triplet_collate_fn(batch):
    """Collate function that extracts text triplets from batch"""
    queries = [item[0] for item in batch]
    positives = [item[1] for item in batch]
    negatives = [item[2] for item in batch]
    return queries, positives, negatives

train_dataloader = DataLoader(triplet_dataset, shuffle=True, batch_size=32, collate_fn=triplet_collate_fn)
print(f"✓ DataLoader created: {len(train_dataloader)} batches of size 32")

In [ ]:
# Fine-tune DistilBERT with Triplet Loss - CORRECTED APPROACH
print("\n" + "="*70)
print("PHASE 2: FINE-TUNING DISTILBERT WITH TRIPLET LOSS")
print("="*70 + "\n")

# CRITICAL: Use the same base model as baseline for fair comparison
model_name = "distilbert-base-uncased"
print(f"[1/4] Loading {model_name}...")

# Load the same model used in baseline
tokenizer = AutoTokenizer.from_pretrained(model_name)
finetuned_model = AutoModel.from_pretrained(model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
finetuned_model = finetuned_model.to(device)
print(f"✓ Model loaded on {device}\n")

# Define optimizer and training parameters
from torch.optim import AdamW
from torch.nn import TripletMarginLoss

optimizer = AdamW(finetuned_model.parameters(), lr=2e-5)
triplet_loss_fn = TripletMarginLoss(margin=0.5)

print(f"[2/4] Setting up training...")
print(f"  - Loss function: TripletMarginLoss (margin=0.5)")
print(f"  - Batch size: 32")
print(f"  - Epochs: 3")
print(f"  - Learning rate: 2e-5")
print(f"  - Total steps: {len(train_dataloader) * 3}\n")

# Training loop
epochs = 3
finetuned_model.train()

print(f"[3/4] Starting fine-tuning...\n")
print("Epoch | Avg Loss | Progress")
print("-" * 40)

for epoch in range(epochs):
    total_loss = 0
    num_batches = 0
    
    for batch_idx, batch in enumerate(train_dataloader):
        # batch is (queries, positives, negatives) from custom collate function
        queries, positives, negatives = batch
        
        # Encode all texts
        query_encoded = tokenizer(queries, padding=True, truncation=True, return_tensors='pt', max_length=256)
        query_encoded = {k: v.to(device) for k, v in query_encoded.items()}
        
        positive_encoded = tokenizer(positives, padding=True, truncation=True, return_tensors='pt', max_length=256)
        positive_encoded = {k: v.to(device) for k, v in positive_encoded.items()}
        
        negative_encoded = tokenizer(negatives, padding=True, truncation=True, return_tensors='pt', max_length=256)
        negative_encoded = {k: v.to(device) for k, v in negative_encoded.items()}
        
        # Get embeddings (gradients ENABLED for backprop)
        query_output = finetuned_model(**query_encoded)
        positive_output = finetuned_model(**positive_encoded)
        negative_output = finetuned_model(**negative_encoded)
        
        query_embeddings = mean_pooling(query_output, query_encoded['attention_mask'])
        positive_embeddings = mean_pooling(positive_output, positive_encoded['attention_mask'])
        negative_embeddings = mean_pooling(negative_output, negative_encoded['attention_mask'])
        
        # Compute triplet loss
        loss = triplet_loss_fn(query_embeddings, positive_embeddings, negative_embeddings)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        if (batch_idx + 1) % max(1, len(train_dataloader) // 5) == 0:
            print(f"{epoch+1}    | {total_loss/num_batches:.4f}    | {(batch_idx+1)*100/len(train_dataloader):.1f}%")
    
    avg_loss = total_loss / num_batches
    print(f"{epoch+1}    | {avg_loss:.4f}    | 100.0%")

finetuned_model.eval()
print(f"\n[4/4] Fine-tuning complete!")
print(f"✓ Model ready for evaluation")


In [ ]:
# Evaluate fine-tuned model
print("\n" + "="*70)
print("PHASE 3: EVALUATION - COMPARING BASELINE vs FINE-TUNED")
print("="*70 + "\n")

def evaluate_finetuned_model(model, tokenizer, dataset, split_name="test"):
    """
    Evaluate fine-tuned AutoModel using same encoding as baseline
    """
    print(f"[1/2] Evaluating model on {split_name} set ({len(dataset)} samples)...")
    print(f"Progress | Hit@1 | Hit@3 | Hit@5")
    print("-" * 35)
    
    # Metrics to track
    hits_at_1 = 0
    hits_at_3 = 0
    hits_at_5 = 0
    total_samples = len(dataset)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    
    def encode_texts(texts):
        """Encode texts using same method as baseline"""
        encoded = tokenizer(texts, padding=True, truncation=True, return_tensors='pt', max_length=256)
        encoded = {k: v.to(device) for k, v in encoded.items()}
        
        with torch.no_grad():
            model_output = model(**encoded)
        
        sentence_embeddings = mean_pooling(model_output, encoded['attention_mask'])
        return torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1).cpu().numpy()
    
    # Process each query
    for idx, example in enumerate(tqdm(dataset, desc="Processing queries")):
        query = example['query']
        candidate_chunks = example['candidate_chunks']
        answer_pos = example['answer_pos']
        
        # Encode query
        query_embedding = encode_texts([query])[0]
        
        # Encode all candidate chunks
        chunk_embeddings = encode_texts(candidate_chunks)
        
        # Compute cosine similarity
        similarities = cosine_similarity([query_embedding], chunk_embeddings)[0]
        
        # Get ranking
        ranked_indices = np.argsort(similarities)[::-1]
        
        # Compute Hit@k
        for rank, idx_pos in enumerate(ranked_indices):
            if idx_pos == answer_pos:
                if rank < 1:
                    hits_at_1 += 1
                if rank < 3:
                    hits_at_3 += 1
                if rank < 5:
                    hits_at_5 += 1
                break
        
        # Print progress
        if (idx + 1) % max(1, total_samples // 10) == 0:
            progress_pct = (idx + 1) / total_samples * 100
            h1 = (hits_at_1 / (idx + 1))
            h3 = (hits_at_3 / (idx + 1))
            h5 = (hits_at_5 / (idx + 1))
            print(f"{progress_pct:5.1f}%  | {h1:.3f} | {h3:.3f} | {h5:.3f}")
    
    # Calculate final metrics
    hit_at_1 = hits_at_1 / total_samples
    hit_at_3 = hits_at_3 / total_samples
    hit_at_5 = hits_at_5 / total_samples
    
    print(f"100.0%  | {hit_at_1:.3f} | {hit_at_3:.3f} | {hit_at_5:.3f}")
    print(f"\n[2/2] Evaluation complete!")
    
    return {
        'hit@1': hit_at_1,
        'hit@3': hit_at_3,
        'hit@5': hit_at_5,
    }

# Evaluate fine-tuned model
finetuned_results = evaluate_finetuned_model(finetuned_model, tokenizer, test_data, 'test')

# Compare results
print("\n" + "="*70)
print("RESULTS COMPARISON: BASELINE vs FINE-TUNED")
print("="*70)
print(f"\n{'Metric':<10} {'Baseline':<15} {'Fine-tuned':<15} {'Improvement':<15}")
print("-" * 55)

for metric in ['hit@1', 'hit@3', 'hit@5']:
    baseline = model_1_results[metric]
    finetuned = finetuned_results[metric]
    improvement = ((finetuned - baseline) / baseline) * 100
    
    print(f"{metric:<10} {baseline:<15.4f} {finetuned:<15.4f} {improvement:>+6.2f}%")

print("="*70)